In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc 
from sklearn.metrics import f1_score
import json

import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.load_data_gait import *
from src.plot_data_gait import *
from src.train_LSTMGait import train_model, predict_trial

In [2]:
base_path = '../datasets/GAIT_signals/dataset/data'

# Training

In [3]:
results = []
results_path = "../checkpoints/grid_search_results.json"

if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)

done_keys = {(r["win_size"], r["stride"], r["process"], r["norm"]) for r in results}

for win_size in [50, 100, 200, 400, 500, 700]:
    for overlap in [0, 0.25, 0.5, 0.75]:
        stride = win_size if overlap == 0 else int(win_size * (1 - overlap))

        for process in ["raw", "free_raw"]:
            for norm in ["window", "subj"]:
                key = (win_size, stride, process, norm)
                if key in done_keys:
                    print(f"Skipping already-done combo: win_size={win_size}, stride={stride}, process={process}, norm={norm}")
                    continue

                print(f"Processing with win_size={win_size}, stride={stride}, process={process}, norm={norm}")
                X, y, subjects = load_dataset_gait(base_path, process=process, sensors=['LB'], window_size=win_size, stride=stride)
                X = X.astype(np.float32)
                y = y.astype(np.int64)

                ckpt_path = f"../checkpoints/{process}_win{win_size}_stride{stride}_norm{norm}"
                trained_model, best_val_f1, test_f1 = train_model(X, y, subjects, model_name="cnnbilstm", norm=norm, epochs=200, lr=0.001, patience=20, out_dir=ckpt_path)

                n_subjects = len(np.unique(subjects))
                n_windows = X.shape[0]

                results.append({
                    "win_size": win_size,
                    "stride": stride,
                    "overlap": overlap,
                    "process": process,
                    "norm": norm,
                    "n_subjects": n_subjects,
                    "n_windows": n_windows,
                    "val_f1": round(float(best_val_f1), 6),
                    "test_f1": round(float(test_f1), 6),
                    "checkpoint": ckpt_path,
                })
                print("")
                print("================================")
                print("")

                with open(results_path, 'w') as f:
                    json.dump(results, f, indent=2)

                del X, y, subjects, trained_model
                gc.collect()

print(f"\nDone. {len(results)} combinations trained.")

Processing with win_size=50, stride=50, process=raw, norm=window
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 62085  |  test: 17793
               →  train: 166 subjects (48324 windows)  |  val: 42 subjects (13761 windows)


Early stopping in epoch 72. Best F1: 0.8892

Val F1: 0.8892  |  Test F1: 0.8587
Checkpoint saved → ../checkpoints/raw_win50_stride50_normwindow/model.pt


Processing with win_size=50, stride=50, process=raw, norm=subj
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 62085  |  test: 17793
               →  train: 166 subjects (48324 windows)  |  val: 42 subjects (13761 windows)


Early stopping in epoch 36. Best F1: 0.8829

Val F1: 0.8829  |  Test F1: 0.8589
Checkpoint saved → ../checkpoints/raw_win50_stride50_normsubj/model.pt


Processing with win_size=50, stride=50, process=free_raw, norm=window
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 62085  |  test: 17793
               →  train: 166 subjects (48324 windows)  |  val: 42 subjects (13761 windows)


Early stopping in epoch 62. Best F1: 0.8883

Val F1: 0.8883  |  Test F1: 0.8598
Checkpoint saved → ../checkpoints/free_raw_win50_stride50_normwindow/model.pt


Processing with win_size=50, stride=50, process=free_raw, norm=subj
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 62085  |  test: 17793
               →  train: 166 subjects (48324 windows)  |  val: 42 subjects (13761 windows)


Early stopping in epoch 66. Best F1: 0.8918

Val F1: 0.8918  |  Test F1: 0.8653
Checkpoint saved → ../checkpoints/free_raw_win50_stride50_normsubj/model.pt


Processing with win_size=50, stride=37, process=raw, norm=window
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 83698  |  test: 24004
               →  train: 166 subjects (65155 windows)  |  val: 42 subjects (18543 windows)


KeyboardInterrupt: 

# Results comparison

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("val_f1", ascending=False).reset_index(drop=True)
results_df

,win_size,stride,overlap,process,norm,n_subjects,n_windows,val_f1,test_f1,checkpoint
0,200,50,0.75,free_raw,window,260,76460,0.858819,0.862067,../checkpoints/free_raw_win200_stride50_normwi...
1,400,300,0.25,raw,window,260,12401,0.858411,0.857302,../checkpoints/raw_win400_stride300_normwindow
2,500,500,0.00,raw,window,260,7440,0.858372,0.860186,../checkpoints/raw_win500_stride500_normwindow
3,400,200,0.50,free_raw,window,260,18275,0.858044,0.859213,../checkpoints/free_raw_win400_stride200_normw...
4,200,50,0.75,raw,window,260,76460,0.857951,0.858330,../checkpoints/raw_win200_stride50_normwindow
...,...,...,...,...,...,...,...,...,...,...
91,50,25,0.50,raw,subj,260,160386,0.834542,0.832416,../checkpoints/raw_win50_stride25_normsubj
92,50,50,0.00,free_raw,window,260,80528,0.833797,0.829877,../checkpoints/free_raw_win50_stride50_normwindow
93,50,50,0.00,raw,window,260,80528,0.832930,0.834731,../checkpoints/raw_win50_stride50_normwindow
94,700,175,0.75,raw,subj,260,18453,0.832497,0.836167,../checkpoints/raw_win700_stride175_normsubj


In [ ]:
best_row = results_df.iloc[0]
print("Best configuration (selected by val_f1):")
print(best_row)
print(f"\n--> Reported test_f1 for this configuration: {best_row['test_f1']:.4f}")

Best configuration (selected by val_f1):
win_size                                                    200
stride                                                       50
overlap                                                    0.75
process                                                free_raw
norm                                                     window
n_subjects                                                  260
n_windows                                                 76460
val_f1                                                 0.858819
test_f1                                                0.862067
checkpoint    ../checkpoints/free_raw_win200_stride50_normwi...
Name: 0, dtype: object

--> Reported test_f1 for this configuration: 0.8621


In [ ]:
pivot = results_df.pivot_table(index=["win_size", "stride"], columns="process", values="val_f1")
pivot["free_raw - raw"] = pivot["free_raw"] - pivot["raw"]
pivot.sort_values("free_raw - raw", ascending=False)

process          free_raw       raw  free_raw - raw
win_size stride                                    
200      50      0.856650  0.849807        0.006843
500      375     0.852168  0.845673        0.006495
700      525     0.850785  0.845079        0.005706
         350     0.849520  0.843842        0.005679
400      300     0.854396  0.848946        0.005450
100      25      0.853435  0.848069        0.005366
700      175     0.845465  0.840545        0.004919
400      400     0.852885  0.848133        0.004752
200      200     0.852428  0.847719        0.004708
500      250     0.850247  0.845603        0.004644
50       25      0.841875  0.837241        0.004633
200      100     0.854307  0.849716        0.004591
500      500     0.854731  0.850302        0.004429
200      150     0.853916  0.849808        0.004107
50       50      0.836682  0.832589        0.004092
         37      0.840183  0.836440        0.003743
100      100     0.846256  0.842631        0.003626
50       12      0.844023  0.840725        0.003298
100      50      0.850490  0.847214        0.003276
         75      0.848286  0.845271        0.003015
400      200     0.852287  0.849362        0.002925
         100     0.852691  0.850260        0.002431
700      700     0.852999  0.850690        0.002309
500      125     0.849401  0.848723        0.000677

In [ ]:
subject_sets = {}
for r in results:
    ckpt_file = os.path.join(r["checkpoint"], "model.pt")
    if not os.path.exists(ckpt_file):
        continue
    import torch
    ckpt = torch.load(ckpt_file, map_location='cpu', weights_only=False)
    subject_sets[(r["win_size"], r["stride"], r["process"])] = set(ckpt["test_subjects"])

reference_key, reference_set = next(iter(subject_sets.items()))
mismatches = {k: v for k, v in subject_sets.items() if v != reference_set}

if mismatches:
    print(f"WARNING: {len(mismatches)} configuration(s) have a different test-subject split than {reference_key}.")
    print("This means n_subjects differs across configs (likely due to short trials being dropped")
    print("at larger window_size), so val_f1/test_f1 are not perfectly comparable across all rows.")
    for k in list(mismatches.keys())[:10]:
        print("  -", k)
else:
    print("OK: all configurations share the exact same test-subject split.")

OK: all configurations share the exact same test-subject split.
